# Module 2 · Lesson 03: Chain-of-Thought Prompting

**Chain-of-Thought (CoT)** prompting asks the model to "think step by step".
This dramatically improves accuracy on reasoning, math, and logic tasks.

## What you will learn
1. Why CoT works (and when it doesn't)
2. Zero-shot CoT ("think step by step")
3. Few-shot CoT (reasoning examples)
4. CoT for **code debugging**
5. CoT for **decision making**
6. Different **trigger phrases**

In [ ]:
# ── Setup ──────────────────────────────────────────────
import os
from pathlib import Path
from dotenv import load_dotenv
from IPython.display import display, Markdown

load_dotenv(Path.cwd().parent / ".env")

from openai import OpenAI
client = OpenAI()

def ask(prompt, system=None, temperature=0, max_tokens=15000):
    msgs = []
    if system:
        msgs.append({"role": "system", "content": system})
    msgs.append({"role": "user", "content": prompt})
    r = client.chat.completions.create(
        model="gpt-4o-mini", messages=msgs,
        temperature=temperature, max_tokens=max_tokens
    )
    return r.choices[0].message.content

if client:
    print("Ready")

✅ Ready


---
## 1. The Problem: Without CoT

Let's see what happens when we ask a tricky math problem *without* CoT:

In [2]:
# ── Without CoT ──────────────────────────────────────
problem = """A store has 15 apples. It sells 7 in the morning and receives a delivery of 12.
Then it sells 5 more in the afternoon. How many apples remain?"""

direct_answer = ask(f"{problem}\n\nAnswer with just the number.")
print(f"Direct answer (no CoT): {direct_answer}")

Direct answer (no CoT): 15


---
## 2. Zero-Shot CoT — "Let's think step by step"

Simply adding **"Let's think step by step"** triggers reasoning:

In [5]:
# ── With CoT ─────────────────────────────────────────
cot_answer = ask(f"{problem}\n\nLet's think step by step.  Use plain text, not LaTeX.")
# cot_answer = ask(f"{problem}\n\nLet's think step by step.")
display(Markdown(f"### Step-by-Step Reasoning\n\n{cot_answer.strip()}"))

### Step-by-Step Reasoning

Let's break it down step by step:

1. The store starts with 15 apples.
2. In the morning, it sells 7 apples. 
   - 15 - 7 = 8 apples remaining.
3. The store then receives a delivery of 12 apples.
   - 8 + 12 = 20 apples now in total.
4. In the afternoon, it sells 5 more apples.
   - 20 - 5 = 15 apples remaining.

So, after all the sales and the delivery, the store has 15 apples left.

In [8]:
problem = """John has 3 boxes. 
Each box contains 5 bags. 
Each bag contains 4 marbles.

He gives away 18 marbles to his friend.

Then he buys 2 more boxes with the same structure.

How many marbles does he have now?"""

In [9]:
direct = ask(f"{problem}\n\nAnswer with just the number.", temperature=0)
print(direct)

62


In [12]:
cot = ask(f"{problem}\n\nLet's think step by step. Do NOT use LaTeX", temperature=0)
display(Markdown(cot.strip()))

Let's break this down step by step.

1. **Calculate the total number of marbles in the initial boxes:**
   - John has 3 boxes.
   - Each box contains 5 bags.
   - Each bag contains 4 marbles.

   First, calculate the total number of marbles in one box:
   - Number of marbles in one box = 5 bags * 4 marbles/bag = 20 marbles.

   Now, calculate the total number of marbles in 3 boxes:
   - Total marbles in 3 boxes = 3 boxes * 20 marbles/box = 60 marbles.

2. **Subtract the marbles he gives away:**
   - John gives away 18 marbles.
   - Remaining marbles = 60 marbles - 18 marbles = 42 marbles.

3. **Calculate the number of marbles in the new boxes he buys:**
   - John buys 2 more boxes.
   - Each of these boxes also contains 5 bags, and each bag contains 4 marbles.

   Calculate the total number of marbles in one new box:
   - Number of marbles in one new box = 5 bags * 4 marbles/bag = 20 marbles.

   Now, calculate the total number of marbles in 2 new boxes:
   - Total marbles in 2 new boxes = 2 boxes * 20 marbles/box = 40 marbles.

4. **Add the marbles from the new boxes to the remaining marbles:**
   - Total marbles now = Remaining marbles + Marbles from new boxes = 42 marbles + 40 marbles = 82 marbles.

So, after all these calculations, John has 82 marbles now.

> 💡 **The research:** Kojima et al. (2022) showed that adding "Let's think step by step" 
> improved accuracy by 10–40% on reasoning benchmarks — with zero examples!

---
## 3. Logical Reasoning

In [6]:
# ── Logical reasoning ────────────────────────────────
logic = """Consider these statements:
1. All programmers drink coffee.
2. Some coffee drinkers are night owls.
3. No night owls wake up early.

Question: Can we conclude that some programmers don't wake up early?

Think through this step by step, examining each logical connection."""

result = ask(logic)
display(Markdown(result))

Let's analyze the statements step by step to see if we can conclude that some programmers don't wake up early.

1. **All programmers drink coffee.**
   - This means that every individual who is a programmer is also a coffee drinker. We can represent this as:
     - If \( P \) is a programmer, then \( C \) (coffee drinker).

2. **Some coffee drinkers are night owls.**
   - This indicates that there is at least one individual who drinks coffee and is also a night owl. We can represent this as:
     - There exists at least one \( C \) such that \( N \) (night owl).

3. **No night owls wake up early.**
   - This means that if someone is a night owl, then they do not wake up early. We can represent this as:
     - If \( N \) (night owl), then not \( E \) (does not wake up early).

Now, let's connect these statements:

- From statement 1, we know that all programmers are coffee drinkers. Therefore, all programmers belong to the group of coffee drinkers.
- From statement 2, since some coffee drinkers are night owls, it follows that there are coffee drinkers who are also night owls.
- Since all programmers are coffee drinkers, it is possible that some of these coffee drinkers (who are night owls) are also programmers.

Now, if we take the coffee drinkers who are night owls (from statement 2) and apply statement 3, we can conclude that those night owls do not wake up early.

Thus, if some coffee drinkers (who are night owls) are indeed programmers, then those specific programmers do not wake up early.

Therefore, we can conclude that **some programmers don't wake up early** based on the logical connections established by the statements.

---
## 4. CoT for Code Debugging

CoT is particularly effective for finding bugs — it forces the model to trace execution:

In [9]:
# ── Code debugging with CoT ──────────────────────────
debug_prompt = """Find the bug in this Python code:

```python
def find_duplicates(lst):
    seen = set()
    duplicates = set()
    for item in lst:
        if item in seen:
            duplicates.add(item)
    seen.add(item)
    return list(duplicates)

# Expected: [2, 3]
# Actual:   []
print(find_duplicates([1, 2, 2, 3, 3, 4]))
```

Trace through the code step by step with the input [1, 2, 2, 3, 3, 4]."""

result = ask(debug_prompt)
display(Markdown(result))

Let's trace through the provided code step by step with the input list `[1, 2, 2, 3, 3, 4]`.

Here is the code again for reference:

```python
def find_duplicates(lst):
    seen = set()
    duplicates = set()
    for item in lst:
        if item in seen:
            duplicates.add(item)
        seen.add(item)
    return list(duplicates)

# Expected: [2, 3]
# Actual:   []
print(find_duplicates([1, 2, 2, 3, 3, 4]))
```

### Step-by-step Execution:

1. **Initialization**:
   - `seen = set()`
   - `duplicates = set()`

2. **First Iteration** (`item = 1`):
   - Check if `1` is in `seen`: `1 not in seen` (False)
   - Since it's not in `seen`, we do not add it to `duplicates`.
   - Add `1` to `seen`: `seen = {1}`

3. **Second Iteration** (`item = 2`):
   - Check if `2` is in `seen`: `2 not in seen` (False)
   - Since it's not in `seen`, we do not add it to `duplicates`.
   - Add `2` to `seen`: `seen = {1, 2}`

4. **Third Iteration** (`item = 2`):
   - Check if `2` is in `seen`: `2 in seen` (True)
   - Since it is in `seen`, we add `2` to `duplicates`: `duplicates = {2}`
   - Add `2` to `seen` again (no effect since it's already there): `seen = {1, 2}`

5. **Fourth Iteration** (`item = 3`):
   - Check if `3` is in `seen`: `3 not in seen` (False)
   - Since it's not in `seen`, we do not add it to `duplicates`.
   - Add `3` to `seen`: `seen = {1, 2, 3}`

6. **Fifth Iteration** (`item = 3`):
   - Check if `3` is in `seen`: `3 in seen` (True)
   - Since it is in `seen`, we add `3` to `duplicates`: `duplicates = {2, 3}`
   - Add `3` to `seen` again (no effect since it's already there): `seen = {1, 2, 3}`

7. **Sixth Iteration** (`item = 4`):
   - Check if `4` is in `seen`: `4 not in seen` (False)
   - Since it's not in `seen`, we do not add it to `duplicates`.
   - Add `4` to `seen`: `seen = {1, 2, 3, 4}`

### Conclusion:
After the loop completes, we have:
- `seen = {1, 2, 3, 4}`
- `duplicates = {2, 3}`

Finally, the function returns `list(duplicates)`, which is `list({2, 3})`. This will return `[2, 3]`, which is the expected output.

### Bug Identification:
The bug in the code is due to the indentation of the line `seen.add(item)`. It is currently outside the `for` loop, which means it only adds the last `item` (which is `4` in this case) to `seen` after the loop has completed. 

To fix the code, the line `seen.add(item)` should be indented to be part of the `for` loop:

```python
def find_duplicates(lst):
    seen = set()
    duplicates = set()
    for item in lst:
        if item in seen:
            duplicates.add(item)
        seen.add(item)  # This line should be indented to be part of the loop
    return list(duplicates)
```

With this correction, the function will work as intended and return the expected output `[2, 3]`.

---
## 5. Few-Shot CoT

Providing **reasoning examples** teaches the model HOW to think, not just what to output:

In [13]:
# ── Few-shot CoT ─────────────────────────────────────
prompt = """Solve word problems step by step.  Use plain text, not LaTeX.

Q: If John has 3 boxes with 5 apples each, and he gives away 4 apples, how many does he have?
A: Step 1: John starts with 3 × 5 = 15 apples.
   Step 2: He gives away 4, so 15 - 4 = 11 apples.
   Answer: 11

Q: A train travels 60 km/h for 2 hours, then 90 km/h for 1 hour. What's the total distance?
A: Step 1: First leg: 60 × 2 = 120 km.
   Step 2: Second leg: 90 × 1 = 90 km.
   Step 3: Total: 120 + 90 = 210 km.
   Answer: 210 km

Q: A rectangle has a perimeter of 30 cm. If the length is twice the width, what is the area?
A:"""

result = ask(prompt)
display(Markdown(f"### Few-Shot CoT Result\n\n{result}"))

### Few-Shot CoT Result

Step 1: Let the width be W cm. Then the length is 2W cm.

Step 2: The formula for the perimeter of a rectangle is P = 2(length + width). So, we have:
2(2W + W) = 30.

Step 3: Simplifying the equation: 2(3W) = 30, which gives us 6W = 30.

Step 4: Dividing both sides by 6, we find W = 5 cm.

Step 5: Now, calculate the length: Length = 2W = 2 × 5 = 10 cm.

Step 6: The area of a rectangle is given by the formula Area = length × width. So, Area = 10 × 5 = 50 cm².

Answer: 50 cm²

---
## 6. CoT for Decision Making

In [15]:
# ── Decision making with CoT ─────────────────────────
decision = """We need to choose a database for our new project.

Requirements:
- Flexible schema (data structure may change)
- High write throughput (IoT sensor data)
- Horizontal scaling needed
- Team has SQL experience
- Budget is limited

Options: PostgreSQL, MongoDB, DynamoDB, TimescaleDB

Analyze each option against our requirements step by step, then recommend one."""

result = ask(decision, max_tokens=1600)
display(Markdown(result))

Let's analyze each of the database options against your requirements step by step:

### 1. PostgreSQL
- **Flexible Schema**: PostgreSQL supports JSONB, which allows for some flexibility in schema design. However, it is primarily a relational database, so it may not be as flexible as NoSQL options.
- **High Write Throughput**: PostgreSQL can handle a decent amount of write throughput, but it may not be as optimized for high-volume write scenarios as some NoSQL databases.
- **Horizontal Scaling**: PostgreSQL traditionally has limitations in horizontal scaling, although there are extensions and tools (like Citus) that can help with this.
- **Team SQL Experience**: Your team will likely be comfortable with PostgreSQL due to their SQL experience.
- **Budget**: PostgreSQL is open-source and free to use, which is beneficial for a limited budget.

### 2. MongoDB
- **Flexible Schema**: MongoDB is a NoSQL database that offers a very flexible schema, allowing for easy changes to the data structure.
- **High Write Throughput**: MongoDB is designed for high write throughput and can handle large volumes of data, making it suitable for IoT sensor data.
- **Horizontal Scaling**: MongoDB supports horizontal scaling natively through sharding, which is beneficial for your needs.
- **Team SQL Experience**: While MongoDB uses a query language that is different from SQL, the team may need to learn some new concepts, but it is generally considered user-friendly.
- **Budget**: MongoDB has a free tier (Community Edition), but costs can increase with the use of managed services or advanced features.

### 3. DynamoDB
- **Flexible Schema**: DynamoDB is a NoSQL database that offers a flexible schema, allowing for easy changes to the data structure.
- **High Write Throughput**: DynamoDB is designed for high write throughput and can handle massive amounts of data, making it suitable for IoT applications.
- **Horizontal Scaling**: DynamoDB automatically scales horizontally, which is a significant advantage for your project.
- **Team SQL Experience**: DynamoDB uses a different query language (part of AWS SDK), so your team may need to adapt, but it does offer a SQL-like query language called PartiQL.
- **Budget**: DynamoDB has a pay-as-you-go pricing model, which can become expensive depending on usage, especially for high write throughput.

### 4. TimescaleDB
- **Flexible Schema**: TimescaleDB is built on PostgreSQL and offers some flexibility with time-series data, but it is still fundamentally a relational database.
- **High Write Throughput**: TimescaleDB is optimized for time-series data and can handle high write throughput effectively.
- **Horizontal Scaling**: TimescaleDB supports horizontal scaling, but it may not be as straightforward as some NoSQL options.
- **Team SQL Experience**: Your team will be comfortable with TimescaleDB due to its SQL compatibility.
- **Budget**: TimescaleDB is open-source and free to use, which is beneficial for a limited budget.

### Recommendation
Considering your requirements:

- **Flexible Schema**: MongoDB and DynamoDB excel here.
- **High Write Throughput**: MongoDB and DynamoDB are both strong contenders.
- **Horizontal Scaling**: MongoDB and DynamoDB are designed for this.
- **Team SQL Experience**: PostgreSQL and TimescaleDB are the most familiar, but MongoDB is also user-friendly.
- **Budget**: PostgreSQL and TimescaleDB are free, while MongoDB has a free tier, and DynamoDB can become costly.

**Final Recommendation**: **MongoDB** is the best fit for your project. It offers the flexibility you need for changing data structures, high write throughput for IoT data, and horizontal scaling capabilities. While your team may need to learn some new query language concepts, MongoDB is generally considered approachable. If budget constraints are a significant concern, you could also consider **TimescaleDB** as a secondary option, especially if your data is primarily time-series based and you want to leverage SQL familiarity.

---
## 7. CoT Trigger Phrases

Different phrases work for different scenarios:

| Trigger | Best For |
|---------|----------|
| "Let's think step by step" | General reasoning |
| "Walk me through your reasoning" | Complex analyses |
| "Show your work" | Math problems |
| "Trace through the code" | Debugging |
| "Consider each option" | Decision making |
| "First, identify the key facts" | Reading comprehension |

In [ ]:
# ── Compare trigger phrases ──
triggers = [
    "Let's think step by step.",
    "Show your work.",
    "Walk me through your reasoning.",
]

problem = "What is 15% of 240?"

for trigger in triggers:
    result = ask(f"{problem} {trigger}", max_tokens=1500)
    display(Markdown(f"### Trigger: \"{trigger}\""))
    # print(f"Answer:  {result[:1000]}...\n")
    display(Markdown(f"### Answer:\n\n{result}"))
    print("-" * 71)

### Trigger: "Let's think step by step."

### Answer:

To find 15% of 240, you can follow these steps:

1. **Convert the percentage to a decimal**: 
   - 15% can be written as a decimal by dividing by 100: 
   \[
   15\% = \frac{15}{100} = 0.15
   \]

2. **Multiply the decimal by the number**: 
   - Now, multiply 0.15 by 240:
   \[
   0.15 \times 240
   \]

3. **Perform the multiplication**:
   - You can do this calculation:
   \[
   0.15 \times 240 = 36
   \]

So, 15% of 240 is **36**.

-----------------------------------------------------------------------


### Trigger: "Show your work."

### Answer:

To find 15% of 240, you can follow these steps:

1. Convert the percentage to a decimal. To do this, divide 15 by 100:
   \[
   15\% = \frac{15}{100} = 0.15
   \]

2. Multiply the decimal by the number (240):
   \[
   0.15 \times 240
   \]

3. Perform the multiplication:
   \[
   0.15 \times 240 = 36
   \]

Therefore, 15% of 240 is **36**.

-----------------------------------------------------------------------


### Trigger: "Walk me through your reasoning."

### Answer:

To find 15% of 240, you can follow these steps:

1. **Convert the percentage to a decimal**: 
   - 15% can be written as a decimal by dividing by 100. 
   - So, 15% = 15 / 100 = 0.15.

2. **Multiply the decimal by the number**: 
   - Now, multiply 0.15 by 240.
   - Calculation: \( 0.15 \times 240 \).

3. **Perform the multiplication**:
   - \( 0.15 \times 240 = 36 \).

So, 15% of 240 is **36**.

-----------------------------------------------------------------------


---
## Key Takeaways 📝

| Concept | Detail |
|---------|--------|
| **Zero-shot CoT** | Add "think step by step" — free accuracy boost |
| **Few-shot CoT** | Show reasoning examples for complex tasks |
| **Best for** | Math, logic, debugging, multi-step problems |
| **Not needed for** | Simple factual questions, translation |
| **Trade-off** | More tokens used (longer output) = higher cost |

---
**Next:** `04_pctf_framework.ipynb` — A systematic framework for writing effective prompts